# 循环神经网络（RNN）

## 1. 为什么需要RNN
在语言模型中，n-gram模型存在问题：

- 只能依赖固定长度的历史（n-1个词）
- n 增大 → 参数数量指数增长（|V|ⁿ）
- 难以建模长期依赖

因此，引入隐变量模型：

P(x_t | x₁, …, x_{t-1}) ≈ P(x_t | h_{t-1})

其中：
h_{t-1} 表示“历史信息的压缩表示”（隐状态）


## 2. 隐状态（Hidden State）

隐状态的更新方式：

h_t = f(x_t, h_{t-1})

含义：
- 输入：当前输入 + 上一时刻记忆
- 输出：新的记忆

本质上隐状态是对“过去所有信息”的总结

## 3. 与多层感知机的区别

MLP无记忆，RNN有记忆

H_t = φ(X_t W_xh + H_{t-1} W_hh + b)

可以处理序列数据，参数在时间上共享


## 4. RNN的计算过程

每个时间步做两件事：

1. 更新隐状态：
   H_t = φ(X_t W_xh + H_{t-1} W_hh + b)

2. 计算输出：
   O_t = H_t W_hq + b_q

## 5. 参数组成

RNN的参数包括：

- W_xh：输入 → 隐状态
- W_hh：隐状态 → 隐状态（记忆传递）
- b_h：隐层偏置
- W_hq：隐状态 → 输出
- b_q：输出层偏置

所有时间步共享同一组参数


## 6. 计算本质（重要理解）


X_t W_xh + H_{t-1} W_hh

等价于：

[X_t, H_{t-1}] · [W_xh; W_hh]

本质上都是拼接输入和历史，做一次线性变换。



In [1]:
import torch

In [2]:
X, W_xh = torch.normal(0, 1, (3, 1)), torch.normal(0, 1, (1, 4))
H, W_hh = torch.normal(0, 1, (3, 4)), torch.normal(0, 1, (4, 4))
torch.matmul(X, W_xh) + torch.matmul(H, W_hh)

tensor([[-1.0126,  0.3018,  1.5892, -0.1891],
        [-2.9083, -0.7019,  2.6262,  1.6424],
        [-1.7268, -1.3116, -1.5055,  1.1696]])

沿列（轴1）拼接矩阵X和H， 沿行（轴0）拼接矩阵W_xh和W_hh。再将这两个拼接的矩阵相乘， 我们得到与上面相同形状的输出矩阵。

In [3]:
torch.matmul(torch.cat((X, H), 1), torch.cat((W_xh, W_hh), 0))

tensor([[-1.0126,  0.3018,  1.5892, -0.1891],
        [-2.9083, -0.7019,  2.6262,  1.6424],
        [-1.7268, -1.3116, -1.5055,  1.1696]])

# 基于RNN的字符级语言模型

## 1. 基本思想

语言模型的目标是：

根据当前和过去的词元，预测下一个词元。

## 2. RNN在语言模型中的作用

RNN通过隐状态来建模历史信息：

h_t = f(x_t, h_{t-1})

每个时间步中，我们输入当前字符，结合过去的隐状态预测下一个字符

输出通过 softmax 得到概率分布：
P(x_{t+1} | x_1, ..., x_t)

## 3. 训练方式

在每个时间步：

1. 计算输出概率（softmax）
2. 与真实标签比较
3. 使用交叉熵损失训练

每个时间步都有损失，整个序列的损失是所有时间步的平均

## 4. 批量处理

实际训练中，batch_size > 1，每个字符用向量表示（embedding）

输入：
X_t ∈ ℝ^{n × d}

其中，n为批量大小，d为特征维度

## 5. 困惑度（Perplexity）

用于评价语言模型的好坏。

Perplexity = exp(平均交叉熵)

即
exp( - (1/n) Σ log P(x_t | x_{<t}) )

困惑度可以理解为模型在预测下一个词时的“平均选择数”。
困惑度越小，模型越好，表示模型对下一个词的“不确定性”越低，等价于“压缩文本的能力”。  
